##  Live Data Extraction (API Ingestion)
To collect the live cultural heritage records, we authenticate with the Europeana REST API using our academic API key. We query the database for the explicit string `"Roman Empire"` using the `rich` profile to pull comprehensive metadata records in a paginated loop.

*Note: Because this api call streams 47,157 dynamic records live from Europeana's infrastructure, we execute it here and dump the raw output directly into a local backup file (`new Roman_Empire_RAW_UNLIMITED.csv`) before proceeding to local transformation.*

In [ ]:
import requests
import pandas as pd
import time
from pathlib import Path

# --- CONFIGURATION ---
SEARCH_TERM = "Roman Empire"
SAVE_PATH = Path.home() / "Downloads" / "Roman_Empire_RAW_UNLIMITED.csv"

def extract_all_raw_data():
    all_results = []
    cursor = "*"  # Starting cursor for deep pagination
    page_count = 0
    
    print(f"🚀 Starting TOTAL extraction for: {SEARCH_TERM}")
    print("No limits applied. This will collect all available metadata.")

    while True:
        url = "https://api.europeana.eu/record/v2/search.json"
        params = {
            "query": SEARCH_TERM,
            "wskey": "apidemo",  # Demo key
            "rows": 100,         # Maximum allowed per request
            "profile": "rich",   # Maximum metadata detail
            "cursor": cursor
        }
        
        try:
            response = requests.get(url, params=params, timeout=60)
            if response.status_code != 200:
                print(f"❌ API stopped at status: {response.status_code}")
                break
                
            data = response.json()
            items = data.get("items", [])
            
            if not items:
                print("🏁 Reached the end of the Europeana database.")
                break
            
            all_results.extend(items)
            page_count += 1
            print(f"Page {page_count}: Total records collected: {len(all_results)}...")
            
            # Check for the next cursor
            next_cursor = data.get("nextCursor")
            if not next_cursor or next_cursor == cursor:
                break
            cursor = next_cursor
            
            # Respect the API rate limits
            time.sleep(0.1)
            
        except Exception as e:
            print(f"⚠️ Error during extraction: {e}")
            break

    # --- RAW PROCESSING ---
    print("\n📦 Finalizing raw dataset (this may take a moment)...")
    
    # json_normalize flattens the nested structures into columns
    df = pd.json_normalize(all_results)
    
    # We must convert lists to strings so the CSV doesn't break
    # But we won't delete any columns or rows
    for col in df.columns:
        df[col] = df[col].apply(lambda x: " | ".join(map(str, x)) if isinstance(x, list) else x)

    # Save exactly what we found
    df.to_csv(SAVE_PATH, index=False, encoding='utf-8-sig')
    
    print(f"\n✅ EXTRACTION COMPLETE!")
    print(f"Total Rows Collected: {len(df)}")
    print(f"Total Columns (Metadata Fields): {len(df.columns)}")
    print(f"File Saved: {SAVE_PATH}")

if __name__ == "__main__":
    extract_all_raw_data()

🚀 Starting TOTAL extraction for: Roman Empire
No limits applied. This will collect all available metadata.
Page 1: Total records collected: 100...
Page 2: Total records collected: 200...
Page 3: Total records collected: 300...
Page 4: Total records collected: 400...
Page 5: Total records collected: 500...
Page 6: Total records collected: 600...
Page 7: Total records collected: 700...
Page 8: Total records collected: 800...
Page 9: Total records collected: 900...
Page 10: Total records collected: 1000...
Page 11: Total records collected: 1100...
Page 12: Total records collected: 1200...
Page 13: Total records collected: 1300...
Page 14: Total records collected: 1400...
Page 15: Total records collected: 1500...
Page 16: Total records collected: 1600...
Page 17: Total records collected: 1700...
Page 18: Total records collected: 1800...
Page 19: Total records collected: 1900...
Page 20: Total records collected: 2000...
Page 21: Total records collected: 2100...
Page 22: Total records collec

### Step 2: Data Transformation and Reduction
Here, we import the raw dataset and filter it down to our target schema. This ensures our final data file is clean, optimized, and small enough to be tracked by version control systems like GitHub.

In [ ]:
import pandas as pd
import os

# 1. Define your file paths with the NEW file name
file_raw = r"C:\Users\Awaba\Downloads\new Roman_Empire_RAW_UNLIMITED.csv"
file_clean = r"C:\Users\Awaba\Downloads\Roman_Empire_Europeana_Clean.csv"

print("⏳ Processing the new raw dataset into the clean version...")

# 2. Load the raw data
df_raw = pd.read_csv(file_raw, low_memory=False)

# 3. Find the score column dynamically to avoid errors
score_cols = [c for c in df_raw.columns if 'score' in c.lower()]
if score_cols:
    score_col_name = score_cols[0]
    df_raw = df_raw.rename(columns={score_col_name: 'score'})
else:
    df_raw['score'] = 0 

# 4. Create the 'quality_category' column based on completeness
def get_quality(score_val):
    try:
        val = float(score_val)
    except:
        val = 0
    if val == 0: 
        return "Low (Minimal Data)"
    elif val < 5: 
        return "Medium"
    else: 
        return "High (Rich Metadata)"

if 'completeness' in df_raw.columns:
    df_raw['quality_category'] = df_raw['completeness'].apply(get_quality)
else:
    df_raw['completeness'] = 0
    df_raw['quality_category'] = "Low (Minimal Data)"

# 5. Handle 'year' column safely
if 'year' not in df_raw.columns:
    year_cols = [c for c in df_raw.columns if 'year' in c.lower()]
    if year_cols:
        df_raw = df_raw.rename(columns={year_cols[0]: 'year'})
    else:
        df_raw['year'] = "Unknown"

# 6. Shorten titles to keep the file size light
if 'title' in df_raw.columns:
    df_raw['title_short'] = df_raw['title'].astype(str).str[:80]
else:
    df_raw['title_short'] = "No Title"

# 7. Select and order the exact columns needed for your submission
final_columns = [
    'country', 'dataProvider', 'language', 'type', 'completeness',
    'quality_category', 'year', 'title_short', 'score'
]

# Ensure all required columns exist
for col in final_columns:
    if col not in df_raw.columns:
        df_raw[col] = "Unknown"

df_clean = df_raw[final_columns]

# 8. Save the final clean file
df_clean.to_csv(file_clean, index=False, encoding='utf-8-sig')

print(f"\n✅ Success! Your clean file is saved at: {file_clean}")
print(f"Total Rows Saved: {len(df_clean)}")